# LAB | Intro to Machine Learning

In [1]:
#Step 0 - import libraries needed for this lab 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Standard seed used throughout the notebook, for reproducibility
SEED = 42

In [2]:
# Load the dataset

spaceship = pd.read_csv(
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
)
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


# Step 1 - EDA

In [3]:
print("Rows:", spaceship.shape[0])
print("Columns:", spaceship.shape[1])

Rows: 8693
Columns: 14


The dataset has **8,693 rows** and **14 columns**. Each row represents one passenger, and the columns include demographic info, cabin/travel details, spending amounts, and the target column `Transported`.

In [4]:
# Check for data types
spaceship.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


From `.info()` we can see:
- There are 7 object (categorical/text) columns, 6 float columns, and 1 boolean column (`Transported`, our target).
- Almost every column has some missing values (fewer than 8693 non-null entries), except `PassengerId` and `Transported`.

In [5]:
# Check for missing values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Missing values range from **179 to 217** per column, which is small relative to the *8,693* total rows (about 2%). This confirms it's safe to simply drop the affected rows without losing much data.

# There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
# Handling missing data
spaceship = spaceship.dropna()

In [7]:
# Check for missing values again
spaceship.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [8]:
# check the new shape
spaceship.shape

(6606, 14)

# Step 2 - Introduction to KNN
K Nearest Neighbors is a distance based algorithm, and requeries all **input data to be numerical.**

Let's only select numerical columns as our features.

In [9]:
# KNN | select only numerical columns as our features.
X = spaceship[
    [
        'Age',
        'RoomService',
        'FoodCourt',
        'ShoppingMall',
        'Spa',
        'VRDeck'
    ]
]
X.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,39.0,0.0,0.0,0.0,0.0,0.0
1,24.0,109.0,9.0,25.0,549.0,44.0
2,58.0,43.0,3576.0,0.0,6715.0,49.0
3,33.0,0.0,1283.0,371.0,3329.0,193.0
4,16.0,303.0,70.0,151.0,565.0,2.0


In [10]:
# Define our target.

features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

X = spaceship[features]

y = spaceship['Transported']

# check x
y.head()

0    False
1     True
2    False
3    False
4     True
Name: Transported, dtype: bool

# Step 3 - Train / Test Split**
Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. **80%** of the data should be in the training set and **20%** in the test set.

In [11]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # text size is 0.2 (80/20 split)
    random_state=SEED    # fixed random_state so the split is reproducible
)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

X_train shape: (5284, 6)
X_test shape:  (1322, 6)
y_train shape: (5284,)
y_test shape:  (1322,)


In [12]:
# Step 4 - Fit the model to your data.
# Initialize KNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

In [13]:
# Step 5 -  Evaluate your model.
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 77.16%

Classification Report:
               precision    recall  f1-score   support

       False       0.79      0.74      0.76       653
        True       0.76      0.80      0.78       669

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322

Confusion Matrix:
 [[483 170]
 [132 537]]


## Conclusion

I successfully developed my first *Machine Learning classification model* using **K-Nearest Neighbors (KNN).**

The workflow included:

1. Loading the Spaceship Titanic dataset
2. Exploring the data
3. Checking data types
4. Checking and handling missing values
5. Selecting numerical features
6. Defining the target variable
7. Splitting the data into training and testing sets
8. Initializing and fitting a KNN classifier
9. Making predictions on unseen data
10. Evaluating the model using accuracy

The accuracy score gives us an indication of how well our KNN model predicts whether passengers were transported achieving an overall accuracy of **77.16%.**